## **Generación con Modelo GPT-4.1**

In [ ]:
import json
from openai import OpenAI
from google.colab import userdata

# ==========================
#   OBTENER API KEY SECRETA
# ==========================
api_key = userdata.get("OPENAI_API_KEY")
if api_key is None:
    raise ValueError("No se encontró OPENAI_API_KEY en Colab Secrets")

# ==========================
#   CLIENTE OPENAI
# ==========================
client = OpenAI(api_key=api_key)

# ==========================
#   PREPROCESAMIENTO (MT-RAG / TASK C)
# ==========================
def get_current_question(task):
    """
    La pregunta real es el último mensaje del usuario
    """
    msgs = task.get("input", [])
    for m in reversed(msgs):
        if m.get("speaker") == "user" and m.get("text"):
            return m["text"]
    return ""

def format_conversation_history(task):
    """
    Historial completo de la conversación
    """
    msgs = task.get("input", [])
    lines = []
    for m in msgs:
        speaker = (m.get("speaker") or "unknown").upper()
        text = m.get("text") or ""
        lines.append(f"{speaker}: {text}")
    return "\n".join(lines)

def format_contexts(task):
    ctxs = task.get("contexts", [])
    parts = []
    for i, c in enumerate(ctxs, start=1):
        text = c.get("text") or ""
        parts.append(f"[Passage {i}] {text}")
    return "\n\n".join(parts)

# ==========================
#   PROMPT (IDÉNTICO, SOLO SIN VARIABLE)
# ==========================
def generate_response(task):
    current_question = get_current_question(task)
    conversation_history = format_conversation_history(task)
    contexts_text = format_contexts(task)

    prompt = f"""
You are an AI assistant participating in a shared task where the goal is to generate responses that closely match the target answers in the dataset.

Your SINGLE objective:
→ Produce responses that match the *style, content, structure, and level of detail* of the target answers.

Follow these rules STRICTLY:

1. STYLE & TONE
   - Neutral, factual, concise.
   - No conversational fillers like “Yes,” “No,” “Sure,” etc.
   - No opinions, no creativity, no storytelling.

2. LEVEL OF DETAIL
   - Include only the small factual extensions typically seen in the dataset.
   - If the target answers split values (e.g., “six teams from each conference”), include that.
   - If the target clarifies acronyms (e.g., "NFL = National Football League"), include it.
   - Do NOT add details that the dataset style does not include.

3. CONTEXT USE
   - Use contexts only if relevant.
   - Multi-turn questions: use history only to disambiguate, never restate.

4. ANSWERABILITY
   - If marked unanswerable, respond EXACTLY:
     "I'm sorry, but I don't have the answer to your question."

5. STRUCTURE MATCHING
   - Direct declarative sentences.
   - No leading words (“Yes,” “No,” “In conclusion”).
   - Small clarifications permitted only when dataset consistently includes them.

6. NO HALLUCINATIONS
   - Only state facts present in context or trivial domain facts consistent with dataset behavior.

Now generate the response.

Current Question: {current_question}
Conversation History:
{conversation_history}

Available Contexts:
{contexts_text}

Give the final answer in the exact style described above.
"""

    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.6,
        max_tokens=256
    )

    return completion.choices[0].message.content.strip()

# ==========================
#   LECTURA / ESCRITURA JSONL
# ==========================
input_path = "rag_taskAC_with_answerability_and_contexts.jsonl"
output_path = "responses_TaskC.jsonl"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for idx, line in enumerate(fin):
        task = json.loads(line)

        gen = generate_response(task)

        task["predictions"] = [{"text": gen}]
        fout.write(json.dumps(task, ensure_ascii=False) + "\n")

        print(f"=== TASK {task.get('task_id', idx)} ===")
        print(f"PREGUNTA ACTUAL: {get_current_question(task)}")
        print(f"RESPUESTA GENERADA: {gen}")
        print("-" * 80)

print("✅ responses_TaskC.jsonl generado correctamente.")


=== TASK 18ef26058d321c5d96ca3ebf8117789e<::>7 ===
PREGUNTA ACTUAL: I mean current EV's battery does not stand for a used car market...how do you think?
RESPUESTA GENERADA: EV (electric vehicle) batteries are warrantied for 8 years or 100,000 miles by law. This warranty is the minimum, and some batteries have lasted much longer, with examples of over 300,000 miles. As electric cars become more widespread, there may be a second-life market for used batteries in static storage, which could offset the cost of new batteries.
--------------------------------------------------------------------------------
=== TASK b3b321e9ea81d1d90e528f85fff72d63<::>8 ===
PREGUNTA ACTUAL: I heard the toolchain is not available in South America.
RESPUESTA GENERADA: Toolchains are available in Sao Paulo, which is in South America.
--------------------------------------------------------------------------------
=== TASK 72013190593248ab20cca034f21ce38a<::>4 ===
PREGUNTA ACTUAL: Are those the only steps?
RESPUE